In [ ]:
! pip install -q ultralytics

In [ ]:
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"Device Name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU'}")

In [ ]:
import yaml

dataset_config = {
    # Pointing to the exact location of your dataset in the read-only input folder
    'path': '/kaggle/input/datasets/dhoesh123/civic-non-coco-dataset/dataset', 
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'nc': 3,
    'names': {
        0: 'Pothole',
        1: 'Crack',
        2: 'Manhole'
    }
}

# Saving the new file into the writable working directory
with open('/kaggle/working/data.yaml', 'w') as f:
    yaml.dump(dataset_config, f, default_flow_style=False)

print("New Kaggle-ready data.yaml created successfully in /kaggle/working/!")

In [ ]:
from ultralytics import RTDETR

def main():
    # 1. Initialize the RT-DETR large model 
    model = RTDETR("rtdetr-l.pt") 

    # 2. Start fine-tuning
    results = model.train(
        data="/kaggle/working/data.yaml", # Pointing to our NEW file
        epochs=50,                        
        imgsz=640,                        
        batch=8,                          
        project="/kaggle/working/road_damage_project", # Saving outputs to the writable folder
        name="rtdetr_run1",               
        seed=42,                          
        deterministic=False, 
        device=0 
    )

if __name__ == '__main__':
    main()

In [ ]:
import shutil

# Zip the entire training run directory
shutil.make_archive('/kaggle/working/rtdetr_delivery_pack', 'zip', '/kaggle/working/road_damage_project/rtdetr_run1-2')
print("Zip archive created: /kaggle/working/rtdetr_delivery_pack.zip")

In [ ]:
from collections import Counter
import glob
import os
import shutil

src_dataset = (
    "/kaggle/input/datasets/dhoesh13/civic-non-coco-dataset/dataset"
)
work_dataset = "/kaggle/working/dataset"

if not os.path.exists(work_dataset):
  print("Copying dataset to writable working directory...")
  shutil.copytree(src_dataset, work_dataset)

# Recursively find ALL label text files anywhere inside the working dataset folder
label_files = glob.glob(
    os.path.join(work_dataset, "**/*.txt"), recursive=True
)
# Filter out data.yaml or other non-annotation txt files if any exist
label_files = [lf for lf in label_files if "images" not in lf and "yaml" not in lf]

print(f"Found {len(label_files)} label files to scan.")

current_counts = Counter()
for lf in label_files:
  with open(lf, "r", encoding="utf-8") as f:
    for line in f:
      parts = line.strip().split()
      if parts:
        try:
          current_counts[int(parts[0])] += 1
        except ValueError:
          continue

print(f"Current counts before balancing: {current_counts}")

target_count = 1500
duplicated_count = 0

for label_file in label_files:
  with open(label_file, "r", encoding="utf-8") as f:
    lines = f.readlines()

  has_pothole = any(line.strip().startswith("0") for line in lines)
  has_manhole = any(line.strip().startswith("2") for line in lines)

  should_duplicate = False
  if has_pothole and current_counts[0] < target_count:
    should_duplicate = True
    current_counts[0] += 1
  elif has_manhole and current_counts[2] < target_count:
    should_duplicate = True
    current_counts[2] += 1

  if should_duplicate:
    base_name = os.path.splitext(os.path.basename(label_file))[0]
    sub_dir = os.path.dirname(label_file)

    # Find the corresponding image directory (replace 'labels' with 'images')
    img_sub_dir = sub_dir.replace("labels", "images")
    img_path = None
    for ext in [".jpg", ".jpeg", ".png", ".JPG", ".PNG"]:
      p = os.path.join(img_sub_dir, base_name + ext)
      if os.path.exists(p):
        img_path = p
        break

    if img_path:
      new_name = base_name + "_balanced"
      new_label = os.path.join(sub_dir, new_name + ".txt")
      new_img = os.path.join(
          img_sub_dir, new_name + os.path.splitext(img_path)[1]
      )

      shutil.copy(label_file, new_label)
      shutil.copy(img_path, new_img)
      duplicated_count += 1

print(
    f"Successfully added {duplicated_count} balanced copies! New counts should"
    " approach target."
)

In [ ]:
from collections import Counter
import glob

# Search recursively anywhere inside the working dataset folder for any .txt file
new_labels = glob.glob(
    "/kaggle/working/dataset/**/*.txt", recursive=True
)
# Filter for label files (ignoring yaml)
new_labels = [lf for lf in new_labels if "yaml" not in lf and "images" not in lf]

new_counts = Counter()
for lf in new_labels:
  with open(lf, "r", encoding="utf-8") as f:
    for line in f:
      parts = line.strip().split()
      if parts:
        try:
          new_counts[int(parts[0])] += 1
        except ValueError:
          continue

print("New Balanced Class Counts:", new_counts)

In [ ]:
import yaml

yaml_path = "/kaggle/working/dataset/dataset/data.yaml"

# 1. Read the existing data.yaml
with open(yaml_path, "r") as f:
  data_config = yaml.safe_load(f)

# 2. Correct the internal paths to point to the working directory
data_config["path"] = "/kaggle/working/dataset/dataset"  # Root dataset folder
data_config["train"] = "images/train"  # Relative to path, or absolute
data_config["val"] = "images/val"

# If there is a test key, make sure it's valid too
if "test" in data_config:
  data_config["test"] = "images/test"

# 3. Save the updated configuration back to data.yaml
with open(yaml_path, "w") as f:
  yaml.dump(data_config, f, sort_keys=False)

print("data.yaml updated successfully with correct working paths!")

In [ ]:
from ultralytics import RTDETR

# 1. Load your previously trained weights instead of rtdetr-l.pt
# (Make sure to update this path to wherever your saved best_finetuned.pt is located)
model = RTDETR("//kaggle/input/models/dhoesh123/m/pytorch/default/1/best_finetuned.pt")

# 2. Train/fine-tune on the new balanced dataset
results = model.train(
    data="/kaggle/working/dataset/dataset/data.yaml",
    epochs=15,  # Fewer epochs are needed since it's fine-tuning!
    imgsz=640,
    batch=8,
    device=0,
)

In [ ]:
import os
import shutil

# Define the source directory where your training run is saved
run_dir = "/kaggle/working/runs/detect/train-4"
output_zip = "/kaggle/working/trained_model_run"

if os.path.exists(run_dir):
  # Create a zip archive of the training run folder
  shutil.make_archive(output_zip, "zip", run_dir)
  print(f"Successfully zipped training run to: {output_zip}.zip")
else:
  print(
      f"Directory {run_dir} not found. Check your run path (it might be train-3"
      " or train-5)."
  )